# Sentiment Analysis: Traditional ML vs LLM-based Approaches

**Course:** Napredna analiza podataka

**Student:** Nevena Djordjevic 2022/0052

**Description:**  
This project compares traditional machine learning models and large language models (LLMs) for sentiment analysis on textual data.


## Project Plan

1. Dataset selection and exploration  
2. Data preprocessing and text cleaning  
3. Feature extraction using TF-IDF  
4. Traditional machine learning model for sentiment classification  
5. Zero-shot sentiment classification using a large language model (LLM)  
6. Few-shot sentiment classification using a large language model (LLM)  
7. Model evaluation and comparison  
8. Discussion and conclusions


In [1]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


## 1. Dataset selection and exploration

In this project, the IMDb Movie Reviews dataset is used for sentiment classification.
The dataset consists of textual movie reviews collected from the IMDb platform, where
each review is labeled with a binary sentiment value: **positive** or **negative**.

The dataset is a well-known benchmark for text classification tasks and is widely used
for evaluating sentiment analysis models. Each data instance contains:
- a text field representing the movie review
- a label indicating the sentiment polarity

This dataset is suitable for comparing traditional machine learning approaches with
modern large language model (LLM) based methods, such as zero-shot and few-shot
classification, which will be used in this research.


In [2]:
import zipfile
import os

zip_path = "/content/imdb.zip"
extract_path = "/content/imdb"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
# Extracting ZIP into the target directory

os.listdir(extract_path)
# Quick check: We're expecting a single CSV here

['IMDB Dataset.csv']

In [3]:
import pandas as pd

df = pd.read_csv("/content/imdb/IMDB Dataset.csv")
# Loading the dataset from the extracted CSV file
df.head()
# Preview of the first few rows to verify columns and data format

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df['sentiment'].value_counts()
# Checking if classes are balanced

,count
sentiment,
positive,25000
negative,25000


In [5]:
from sklearn.model_selection import train_test_split

X = df['review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 2. Data preprocessing and text cleaning

Textual data often contains noise and inconsistencies that can negatively
affect the performance of machine learning models. Therefore, a preprocessing
step is applied to normalize the text and reduce irrelevant variability.

In this project, a minimal and controlled text cleaning procedure is used in
order to preserve as much sentiment-related information as possible. The
following preprocessing steps are applied:
- removal of HTML line break tags (`<br />`),
- conversion of text to lowercase,
- normalization of whitespace.

More aggressive preprocessing steps such as stop-word removal, stemming, or
lemmatization are intentionally not applied at this stage, since they may
remove or distort information relevant for sentiment classification.

This preprocessing strategy ensures a balance between noise reduction and
information preservation, which is particularly important when comparing
traditional machine learning models with large language model (LLM) based
approaches.

Why not stemming or lemmatization?
Because sentiment can be expressed through word forms and intensity, and
aggressive normalization might remove useful signal.

In [6]:
# Checking for missing values in the key columns
df[["review", "sentiment"]].isna().sum()

,0
review,0
sentiment,0


In [7]:
# Quick look at review length distribution (number of characters)
df["review_len"] = df["review"].str.len()

df["review_len"].describe()

,review_len
count,50000.000000
mean,1309.431020
std,989.728014
min,32.000000
25%,699.000000
50%,970.000000
75%,1590.250000
max,13704.000000


In [8]:
import re

def clean_text(text: str) -> str:
    """
    Basic text cleaning for IMDb reviews.
    - Remove HTML line breaks (<br />)
    - Remove extra whitespace
    - Lowercase
    Note: We keep punctuation for now (can help sentiment in some cases).
    """
    text = re.sub(r"<br\s*/?>", " ", text)      # remove HTML breaks
    text = text.lower()                        # lowercase
    text = re.sub(r"\s+", " ", text).strip()   # normalize whitespace
    return text

# Apply cleaning to train/test text
X_train_clean = X_train.apply(clean_text)
X_test_clean = X_test.apply(clean_text)

# Sanity check: show before/after for one example
print("BEFORE:\n", X_train.iloc[0][:300], "\n")
print("AFTER:\n", X_train_clean.iloc[0][:300])

BEFORE:
 I caught this little gem totally by accident back in 1980 or '81. I was at a revival theatre to see two old silly sci-fi movies. The theatre was packed full and (with no warning) they showed a bunch of sci-fi short spoofs (to get us in the mood). Most were somewhat amusing but THIS came on and, with 

AFTER:
 i caught this little gem totally by accident back in 1980 or '81. i was at a revival theatre to see two old silly sci-fi movies. the theatre was packed full and (with no warning) they showed a bunch of sci-fi short spoofs (to get us in the mood). most were somewhat amusing but this came on and, with


## 3. Feature extraction using TF-IDF

Machine learning algorithms require numerical input, therefore textual data
must be transformed into a numeric representation before model training.

In this project, text data is represented using the TF-IDF (Term Frequency –
Inverse Document Frequency) approach. TF-IDF assigns a weight to each word
based on:
- how frequently the word appears in a document (term frequency),
- how rare the word is across the entire corpus (inverse document frequency).

This representation has several advantages for sentiment classification:
- it reduces the impact of very common words (e.g. "the", "movie"),
- it emphasizes words that are more informative for sentiment,
- it is computationally efficient and well-suited for linear classifiers.

TF-IDF is a widely used baseline in text classification tasks and provides a
strong and interpretable foundation for comparison with LLM-based approaches.

Why not word embeddings?
TF-IDF is chosen as a strong and interpretable baseline, which allows a fair comparison with zero-shot and few-shot LLM approaches without introducing additional pretrained semantic knowledge.

### 3.1 TF-IDF vectorization


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,      # limit vocabulary size for efficiency
    ngram_range=(1, 2),      # use unigrams and bigrams
    min_df=5                 # ignore very rare words
)

In [10]:
# Fit TF-IDF on training data only -> no data leakage
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_clean)

# Transforming test data using the fitted vectorizer
X_test_tfidf = tfidf_vectorizer.transform(X_test_clean)

In [11]:
X_train_tfidf.shape, X_test_tfidf.shape

((40000, 10000), (10000, 10000))

In [12]:
# Inspect a few feature names
tfidf_vectorizer.get_feature_names_out()[:20]

array(['00', '000', '10', '10 10', '10 for', '10 minutes', '10 out',
       '10 stars', '10 years', '100', '11', '12', '13', '14', '15',
       '15 minutes', '16', '17', '18', '1930'], dtype=object)

## 4. Traditional machine learning model for sentiment classification

In this section, a traditional supervised machine learning approach is applied to the sentiment classification task.
The textual data, previously transformed into numerical representations using the TF-IDF method, is used as input to a classical classification algorithm.

Logistic Regression is selected as the baseline model due to its simplicity, interpretability, and strong performance on high-dimensional sparse text representations such as TF-IDF vectors.
Despite its linear nature, Logistic Regression is widely used as a competitive baseline in sentiment analysis tasks and serves as a strong point of comparison against modern large language model (LLM)-based approaches.

The model is trained on the training subset and evaluated on a held-out test set using standard classification metrics.
This approach provides a clear reference point for assessing the effectiveness of zero-shot and few-shot LLM-based sentiment classification methods in later sections.

In [13]:
# Import machine learning model and evaluation utilities
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [14]:
# Initialize Logistic Regression model
log_reg_model = LogisticRegression(
    max_iter=1000,      # increase iterations to ensure convergence
    random_state=42,
    n_jobs=-1           # use all available CPU cores
)

In [15]:
# Train the Logistic Regression model on TF-IDF features
log_reg_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42)

In [16]:
# Predict sentiment labels on the test set
y_pred = log_reg_model.predict(X_test_tfidf)

In [17]:
# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label="positive")
recall = recall_score(y_test, y_pred, pos_label="positive")
f1 = f1_score(y_test, y_pred, pos_label="positive")

accuracy, precision, recall, f1

(0.9013, 0.8958374432826988, 0.9082, 0.9019763630946469)

In [18]:
# Detailed classification report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negative       0.91      0.89      0.90      5000
    positive       0.90      0.91      0.90      5000

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



**Results and interpretation – Traditional ML model (Logistic Regression)**

In this section, we analyze the performance of the traditional machine learning approach based on TF-IDF feature representation and Logistic Regression for sentiment classification.

The model was trained on 40,000 movie reviews and evaluated on a held-out test set of 10,000 reviews, ensuring a fair and unbiased evaluation.

**Quantitative results**

The obtained evaluation metrics are:

Accuracy: 0.901

Precision (positive class): 0.896

Recall (positive class): 0.908

F1-score (positive class): 0.902

The detailed classification report shows balanced performance across both sentiment classes:

The model achieves similar precision and recall for both positive and negative reviews, indicating no significant bias toward a particular class.

The macro average and weighted average F1-scores are both approximately 0.90, confirming stable performance across classes.

**Interpretation**

These results demonstrate that the traditional TF-IDF + Logistic Regression pipeline is a strong baseline model for sentiment classification:

TF-IDF effectively captures informative word and n-gram patterns relevant to sentiment polarity.

Logistic Regression performs well in high-dimensional sparse feature spaces, making it a suitable choice for text classification tasks.

The achieved F1-score above 0.90 indicates a high level of predictive performance, despite the model’s relative simplicity.

**Importance for further comparison**

The strong performance of this classical machine learning model is particularly important for this study, as it establishes a competitive baseline against which zero-shot and few-shot LLM-based approaches will be compared in the following sections.

Any performance gains achieved by LLM-based methods must therefore be interpreted in light of:

the absence of task-specific training,

increased computational cost,

and reduced interpretability compared to traditional ML models.

## 5. Zero-shot sentiment classification using a Large Language Model (LLM)

Zero-shot classification represents a paradigm in which a model performs a classification task **without being explicitly trained on labeled examples from the target dataset**. Instead of learning from task-specific training data, the model relies on previously acquired linguistic and semantic knowledge obtained during large-scale pretraining.

In the context of sentiment analysis, a zero-shot approach allows a large language model (LLM) to assign sentiment labels (e.g., *positive* or *negative*) to text inputs based solely on a natural language task description and a predefined set of candidate labels.

This approach offers several advantages:
- eliminates the need for labeled training data,
- enables rapid adaptation to new tasks and domains,
- significantly reduces development time.

However, zero-shot classification may also have limitations, such as:
- lower performance compared to supervised models trained on task-specific data,
- sensitivity to prompt formulation and label wording,
- higher computational cost.

In this section, we evaluate the effectiveness of a zero-shot sentiment classification approach using a pretrained large language model and compare its performance with the previously implemented traditional machine learning model.

The zero-shot sentiment classification is implemented using the Hugging Face Transformers library, which provides a high-level interface for working with pretrained transformer models. Specifically, the zero-shot classification pipeline is used, allowing the task to be defined through natural language prompts and candidate labels without explicit model training. This abstraction simplifies model usage while preserving access to state-of-the-art pretrained architectures.


In [19]:
!pip install -q transformers torch

In [20]:
from transformers import pipeline
import numpy as np

In [21]:
# Initialize zero-shot classification pipeline
zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [22]:
# Use a random subset of the test data for a fairer quick evaluation
N_SAMPLES = 200
sample_idx = y_test.sample(n=N_SAMPLES, random_state=42).index

X_test_sample = X_test_clean.loc[sample_idx].tolist()
y_test_sample = y_test.loc[sample_idx].tolist()

In [23]:
# Candidate labels for zero-shot classification
candidate_labels = ["positive", "negative"]

hypothesis_template = "This movie review is {}."

zero_shot_preds = []

for text in X_test_sample:
    out = zero_shot_classifier(
        text,
        candidate_labels=candidate_labels,
        hypothesis_template=hypothesis_template,
        multi_label=False
    )
    # out["labels"] is sorted by score descending -> first is predicted label
    zero_shot_preds.append(out["labels"][0])

zero_shot_preds[:10], y_test_sample[:10]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


(['negative',
  'positive',
  'negative',
  'negative',
  'positive',
  'negative',
  'negative',
  'positive',
  'negative',
  'positive'],
 ['negative',
  'positive',
  'negative',
  'negative',
  'negative',
  'negative',
  'negative',
  'positive',
  'negative',
  'positive'])

In [24]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

zs_acc = accuracy_score(y_test_sample, zero_shot_preds)
zs_prec = precision_score(y_test_sample, zero_shot_preds, pos_label="positive")
zs_rec = recall_score(y_test_sample, zero_shot_preds, pos_label="positive")
zs_f1 = f1_score(y_test_sample, zero_shot_preds, pos_label="positive")

zs_acc, zs_prec, zs_rec, zs_f1

(0.945, 0.9259259259259259, 0.970873786407767, 0.9478672985781991)

In [25]:
print(classification_report(y_test_sample, zero_shot_preds))

              precision    recall  f1-score   support

    negative       0.97      0.92      0.94        97
    positive       0.93      0.97      0.95       103

    accuracy                           0.94       200
   macro avg       0.95      0.94      0.94       200
weighted avg       0.95      0.94      0.94       200



**Zero-shot sentiment classification (Evaluation)**

Zero-shot sentiment classification was evaluated on a randomly selected subset of 200 test samples due to the computational cost of large language models. The facebook/bart-large-mnli model is a natural language inference (NLI) model that is commonly used for zero-shot text classification by reformulating the task as an entailment problem using a hypothesis template. The model was used without any task-specific fine-tuning.

Despite the absence of supervised training on the IMDb dataset, the model achieved an accuracy of 94.5%, with an F1-score of 0.95 for the positive class. These results indicate that large pre-trained language models are capable of capturing sentiment-related semantic patterns directly from natural language, even in a zero-shot setting.

The balanced performance across both sentiment classes, as reflected by the macro-averaged F1-score of 0.94, suggests that the zero-shot model generalizes well and does not favor a specific sentiment polarity. This further supports the suitability of large pre-trained language models for sentiment analysis tasks in scenarios where labeled data is unavailable.

Important note:
The comparison should be interpreted with caution, as the zero-shot model was evaluated on a smaller subset of the test data due to computational constraints. However, the results still provide valuable insight into the trade-off between performance and computational cost.

## 6. Few-shot sentiment classification using a large language model (LLM)

Few-shot sentiment classification extends the zero-shot approach by providing the model with a small number of labeled examples directly in the prompt. Instead of relying only on general language understanding, the model can use these examples as guidance for how the task should be solved and how the labels should be interpreted in this specific context.

In this project, few-shot classification is implemented by selecting a small, balanced set of IMDb reviews and embedding them into the prompt along with the new, unseen review. The model is then asked to output only one of the predefined labels (positive or negative). This setup allows us to evaluate whether a limited amount of in-context supervision improves performance compared to the zero-shot setting, and how few-shot results compare with the traditional TF-IDF + Logistic Regression baseline.

Because LLM inference is computationally expensive, the few-shot approach is evaluated on the same limited subset of 200 test samples used in the zero-shot experiment. This ensures a fair and controlled comparison between the two LLM-based methods, allowing observed performance differences to be attributed primarily to the use of in-context examples rather than dataset size.

Few-shot classification is highly sensitive to prompt design, including the length of the input text and the selection of in-context examples provided to the model.

In [26]:
!pip install -q transformers sentencepiece accelerate

In [27]:
import re
import random
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

MODEL_NAME = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

device

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

device(type='cuda')

In [28]:
def build_fewshot_examples(X_train_clean, y_train, n_pos=2, n_neg=2, seed=42, max_chars=400):
    rng = np.random.default_rng(seed)

    pos_idx = np.where(np.array(y_train) == "positive")[0]
    neg_idx = np.where(np.array(y_train) == "negative")[0]

    # Randomly sample the desired number of examples from each class
    chosen_pos = rng.choice(pos_idx, size=n_pos, replace=False)
    chosen_neg = rng.choice(neg_idx, size=n_neg, replace=False)

    examples = []
    for i in chosen_pos:
        txt = str(X_train_clean.iloc[i])[:max_chars]
        examples.append((txt, "positive"))
    for i in chosen_neg:
        txt = str(X_train_clean.iloc[i])[:max_chars]
        examples.append((txt, "negative"))

    # Shuffle examples to avoid fixed class ordering in the prompt
    rng.shuffle(examples)
    return examples

# Build a balanced few-shot example set (2 positive + 2 negative)
# Shorter text length is used to reduce prompt size and avoid token overflow
fewshot_examples = build_fewshot_examples(X_train_clean, y_train, n_pos=2, n_neg=2, seed=42, max_chars=250)
fewshot_examples

[('"crossfire" is remembered not so much for the fact that its three stars all had the first name "robert" but as being one of the first hollywood films to deal with anti-semitism. the story opens with the murder in silhouette of a man whom we later lea',
  'positive'),
 ("man stop making sequels to great movies. the original was a great movie that was over the top with fights,sex,and one of the coolest characters that graced the screen in the 90's. the story is believable as if your been to bars in the outs of the sou",
  'negative'),
 ('yet another british romantic comedy which audiences all over the world seem to have a ravenous appetite for. this feeble effort is an unintentional parody of the genre - all the classic clichéd scenes are here from ridiculously elaborate misunderstan',
  'negative'),
 ('robert stack never really got over losing a best supporting actor oscar for his role as kyle in "written on the wind" to anthony quinn\'s 12-minute performance in "lust for life." stac

In [29]:
# Constructing a few-shot prompt for sentiment classification.
def make_prompt(review_text: str, examples, max_review_chars=450) -> str:
    review_text = str(review_text)[:max_review_chars]

    prompt = (
        "You are a sentiment classifier.\n"
        "Instruction: Answer with exactly one word: positive or negative.\n"
        "Do not explain.\n\n"
        "Examples:\n"
    )
    # Adding few-shot examples to the prompt
    for ex_text, ex_label in examples:
        prompt += f'Review: "{ex_text}"\nLabel: {ex_label}\n\n'

    prompt += (
        "Now classify this review.\n"
        f'Review: "{review_text}"\n'
        "Label:"
    )
    return prompt

# Normalizing the raw model output into a valid sentiment label.
def normalize_label(generated: str) -> str:
    s = generated.strip().lower()
    # Handling cases where both labels appear in the output
    if "positive" in s and "negative" in s:
        return "positive" if s.find("positive") < s.find("negative") else "negative"

    if "positive" in s:
        return "positive"
    if "negative" in s:
        return "negative"
    return "unknown"

In [30]:
import torch

def score_label(prompt: str, label: str) -> float:
    """
    Returns loss for generating `label` given `prompt`.
    Lower loss = more likely label.
    """
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with tokenizer.as_target_tokenizer():
        targets = tokenizer(label, return_tensors="pt").input_ids.to(device)

    with torch.no_grad():
        out = model(**inputs, labels=targets)
        return out.loss.item()

def flan_predict_label_by_scoring(review_text: str, examples) -> str:
    prompt = make_prompt(review_text, examples)

    # Encode input (prompt)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    labels = ["positive", "negative"]
    losses = {}

    with torch.no_grad():
        for label in labels:
            # Encode target label
            target_ids = tokenizer(
                label,
                return_tensors="pt",
                truncation=True
            ).input_ids.to(device)

            outputs = model(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                labels=target_ids
            )

            losses[label] = outputs.loss.item()

    # Pick label with lower loss
    return min(losses, key=losses.get)

In [31]:
# sanity check
def flan_predict_label(review_text: str, examples) -> str:
    prompt = make_prompt(review_text, examples)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False
        )

    gen = tokenizer.decode(out_ids[0], skip_special_tokens=True)
    return normalize_label(gen)

# quick test
test_pred = flan_predict_label(X_test_clean.loc[sample_idx[0]], fewshot_examples)
test_pred

'negative'

In [32]:
from tqdm.auto import tqdm

# Use the same evaluation subset as zero-shot (fair comparison)

fewshot_preds = []

for text in tqdm(X_test_sample, desc="Few-shot (FLAN-T5) predicting"):
    pred = flan_predict_label_by_scoring(text, fewshot_examples)
    fewshot_preds.append(pred)

# Quick check
fewshot_preds[:10], y_test_sample[:10]

Few-shot (FLAN-T5) predicting:   0%|          | 0/200 [00:00<?, ?it/s]

(['negative',
  'positive',
  'negative',
  'negative',
  'positive',
  'negative',
  'negative',
  'positive',
  'negative',
  'positive'],
 ['negative',
  'positive',
  'negative',
  'negative',
  'negative',
  'negative',
  'negative',
  'positive',
  'negative',
  'positive'])

In [33]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Filter out "unknown" if it ever appears
valid_idx = [i for i, p in enumerate(fewshot_preds) if p in ["positive", "negative"]]

y_true_valid = [y_test_sample[i] for i in valid_idx]
y_pred_valid = [fewshot_preds[i] for i in valid_idx]

print("Valid samples used:", len(valid_idx), "/", len(y_test_sample))

fs_acc = accuracy_score(y_true_valid, y_pred_valid)
fs_prec = precision_score(y_true_valid, y_pred_valid, pos_label="positive", zero_division=0)
fs_rec = recall_score(y_true_valid, y_pred_valid, pos_label="positive", zero_division=0)
fs_f1 = f1_score(y_true_valid, y_pred_valid, pos_label="positive", zero_division=0)

fs_acc, fs_prec, fs_rec, fs_f1

Valid samples used: 200 / 200


(0.885, 0.8921568627450981, 0.883495145631068, 0.8878048780487805)

In [34]:
print(classification_report(y_true_valid, y_pred_valid, zero_division=0))

              precision    recall  f1-score   support

    negative       0.88      0.89      0.88        97
    positive       0.89      0.88      0.89       103

    accuracy                           0.89       200
   macro avg       0.88      0.89      0.88       200
weighted avg       0.89      0.89      0.89       200



**Few-shot sentiment classification (Evaluation)**

Few-shot sentiment classification was evaluated on the same randomly selected subset of 200 test samples used in the zero-shot experiment, in order to ensure a fair and controlled comparison between LLM-based approaches. The google/flan-t5-large model was employed in an instruction-following setup, where a small number of labeled examples were provided directly in the prompt.

Specifically, a 2+2 few-shot configuration was used, consisting of two positive and two negative IMDb reviews sampled from the training set. To reduce prompt noise and avoid exceeding the model’s effective context window, each example review was truncated to 250 characters, which empirically resulted in more stable predictions compared to longer examples. The model was instructed to output only one of the predefined labels (positive or negative).

Unlike the zero-shot approach, which relies on natural language inference, few-shot prediction was performed using a label scoring strategy, where the likelihood of each candidate label was computed and the label with the lower loss was selected. This method avoids free-form text generation and improves output consistency.

**Quantitative results**

The few-shot FLAN-T5 model achieved the following performance on the evaluation subset:

Accuracy: 0.89

Precision (positive class): 0.89

Recall (positive class): 0.88

F1-score (positive class): 0.89

The detailed classification report indicates balanced performance across sentiment classes, with similar precision and recall values for both positive and negative reviews. All 200 test samples were successfully classified without producing undefined or ambiguous labels.

**Interpretation**

The results demonstrate that few-shot in-context learning can significantly improve over naive prompt-based classification and yields robust, balanced predictions even with a very limited number of labeled examples. However, despite this improvement, the few-shot FLAN-T5 approach does not outperform the zero-shot BART-based classifier, which achieved higher accuracy and F1-score in the previous section.

This outcome highlights several important observations:

Few-shot performance is highly sensitive to prompt design, example selection, and input length.

Providing additional examples does not guarantee superior performance compared to a well-aligned zero-shot model.

Large language models trained for instruction-following can generalize effectively, but task alignment and model architecture play a crucial role.

Overall, the few-shot experiment confirms that while in-context supervision can enhance LLM performance, it does not universally surpass zero-shot or traditional supervised methods, particularly when computational constraints and prompt limitations are taken into account.

##7. Model evaluation and comparison

In this section, the performance of all implemented sentiment classification approaches is systematically compared. The goal of this comparison is to evaluate the trade-offs between traditional supervised machine learning models and modern large language model (LLM) based approaches, with respect to predictive performance, data requirements, computational cost, and practical usability.

The following models were evaluated:

TF-IDF + Logistic Regression (supervised baseline),

Zero-shot LLM classification using facebook/bart-large-mnli,

Few-shot LLM classification using google/flan-t5-large.

To ensure fairness, all models were evaluated using standard classification metrics, including accuracy, precision, recall, and F1-score for the positive class. The LLM-based approaches were evaluated on a randomly selected subset of 200 test samples due to computational constraints, while the traditional model was evaluated on the full test set.

**Quantitative comparison**

The table below summarizes the key evaluation results:

| Model                         | Training data                         | Accuracy | F1-score (positive) |
|------------------------------|----------------------------------------|----------|---------------------|
| TF-IDF + Logistic Regression | Full training set (40,000 samples)     | ~0.90    | ~0.90               |
| Zero-shot (BART-MNLI)        | No task-specific training              | **0.95** | **0.95**            |
| Few-shot (FLAN-T5, 2+2)      | 4 in-context examples                  | 0.89     | 0.89                |


**Performance analysis**

The traditional TF-IDF + Logistic Regression model serves as a strong supervised baseline, achieving an F1-score above 0.90 on the full test set. This confirms that classical machine learning methods remain highly effective for sentiment analysis when sufficient labeled data is available.

The zero-shot LLM approach demonstrates surprisingly strong performance, outperforming even the supervised baseline on the evaluated subset. Without any task-specific fine-tuning, the BART-based zero-shot model achieves the highest accuracy and F1-score among all evaluated methods. This result highlights the effectiveness of large pretrained language models in capturing sentiment-related semantic patterns through natural language inference.

The few-shot FLAN-T5 approach significantly improves upon naive prompt-based classification and achieves balanced performance across classes. However, despite the inclusion of labeled examples in the prompt, its performance remains slightly below that of the zero-shot model. This suggests that few-shot performance is highly sensitive to prompt design, example selection, and context length, and that additional in-context supervision does not necessarily guarantee superior results.

**Computational cost and practicality**

While LLM-based approaches show strong performance, they introduce substantially higher computational costs compared to the traditional machine learning model. Inference with large language models is slower, requires specialized hardware (GPU), and limits the number of samples that can be evaluated efficiently.

In contrast, the TF-IDF + Logistic Regression pipeline is computationally efficient, easily scalable, and highly interpretable, making it well-suited for real-world deployment scenarios where labeled data is available.

**Overall comparison and insights**

This comparative evaluation leads to several important conclusions:

Traditional supervised models remain competitive and reliable when labeled data is abundant.

Zero-shot LLMs can achieve state-of-the-art performance without any task-specific training.

Few-shot LLMs benefit from in-context examples but are sensitive to prompt construction and context limitations.

Performance gains of LLM-based methods must be weighed against their higher computational cost and reduced interpretability.

Overall, the results demonstrate that there is no universally superior approach; instead, the optimal choice depends on data availability, computational resources, and application requirements.

##8. Discussion and Conclusions

This project explored sentiment classification on the IMDb Movie Reviews dataset by comparing traditional supervised machine learning methods with modern large language model (LLM) based approaches, including zero-shot and few-shot learning paradigms.

The results demonstrate that classical machine learning models remain highly competitive when sufficient labeled data is available. The TF-IDF + Logistic Regression pipeline achieved strong and stable performance, confirming its effectiveness as a baseline approach for text classification tasks. Its advantages include computational efficiency, scalability, and interpretability, making it suitable for many real-world applications.

At the same time, the experiments show that large pre-trained language models can perform sentiment classification effectively without task-specific training. The zero-shot model achieved the highest performance among all evaluated methods, highlighting the ability of LLMs to generalize semantic knowledge acquired during pretraining to new tasks through natural language instructions alone.

The few-shot experiments provided additional insights into the behavior of instruction-following models. While few-shot prompting improved performance compared to naive prompt-based classification, it did not consistently outperform the zero-shot approach. This suggests that few-shot learning is highly sensitive to prompt design, example selection, and context length. In some cases, additional in-context examples may introduce noise or reduce the effective context available for the input text.

From a practical perspective, the choice of approach depends strongly on the application context. LLM-based methods offer rapid prototyping and eliminate the need for labeled training data, but they come with higher computational costs, longer inference times, and reduced transparency. In contrast, traditional supervised models require labeled data but are easier to deploy and interpret.

**Limitations**

Several limitations should be considered when interpreting the results:

The zero-shot and few-shot models were evaluated on a limited subset of the test data due to computational constraints.

Few-shot performance may vary depending on the selected examples and prompt formulation.

No fine-tuning of LLMs was performed, as the focus was on in-context learning rather than supervised adaptation.

**Final conclusion**

In conclusion, this study demonstrates that sentiment classification can be effectively approached using a wide range of techniques, from traditional machine learning models to large language models operating in zero-shot and few-shot settings. While LLMs offer remarkable flexibility and strong performance without task-specific training, traditional supervised models remain a robust and efficient solution when labeled data is available. The results emphasize the importance of selecting a model based on practical constraints rather than performance alone.